# WhiteBoxProbe — Real Model Validation on MuSiQue n=500

Validates `WhiteBoxProbe` (hidden-state uncertainty detection) on Llama-3-8B-Instruct using
the 500 MuSiQue chains from exp_musique_expand (372 wrong, 128 correct).

**Goal:** AUROC ≥ 0.70 on real model hidden states → WhiteBoxProbe is production-ready for MuSiQue.

**Hardware:** T4 (Colab free tier, 16GB VRAM) is sufficient with 4-bit quantization.

**Steps:**
1. Install deps
2. Upload `musique_chains_200_499.json` from your local machine
3. Load Llama-3-8B in 4-bit
4. Extract hidden states + attention entropy for each chain
5. Train LogReg probe on 70% split, evaluate on 30%
6. Report AUROC + 95% CI
7. Download trained probe pkl

**Cost:** Free (Colab T4). Runtime: ~45 minutes.

In [ ]:
# Cell 1: Install dependencies (Colab Python 3.12 + CUDA 12.x compatible)
# Fixes: bitsandbytes CUDA binary not found + triton.ops missing
!pip install -q "bitsandbytes>=0.45.0"
!pip install -q "transformers>=4.44.0" "accelerate>=0.34.0" "scikit-learn>=1.5.0" "llm-guard-kit==0.24.0"

# Verify bitsandbytes has CUDA support
import importlib, sys
if importlib.util.find_spec("bitsandbytes") is not None:
    import bitsandbytes as bnb
    print(f"bitsandbytes {bnb.__version__} loaded")
    try:
        import bitsandbytes.cuda_setup.main as cs
        print("  CUDA setup: OK")
    except Exception as e:
        print(f"  CUDA setup warning: {e}")

In [ ]:
# Cell 2: Upload chain data
# Upload musique_chains_200_499.json from:
#   results/exp_musique_expand/musique_chains_200_499.json   (300 chains)
# and musique_chains.json from:
#   results/exp156_live_crossdomain/musique_chains.json       (200 chains)

from google.colab import files
print('Upload musique_chains.json (200 chains from exp156):')
uploaded1 = files.upload()
print('Upload musique_chains_200_499.json (300 chains from exp_musique_expand):')
uploaded2 = files.upload()

In [ ]:
# Cell 3: Load chains
import json
chains_200 = json.load(open('musique_chains.json'))
chains_300 = json.load(open('musique_chains_200_499.json'))
all_chains = chains_200 + chains_300
print(f'Total chains: {len(all_chains)}')
n_wrong = sum(1 for c in all_chains if not c.get('correct', True))
print(f'Wrong: {n_wrong}/{len(all_chains)} ({100*n_wrong/len(all_chains):.1f}%)')
print(f'Chain keys: {list(all_chains[0].keys())}')

In [ ]:
# Cell 4: Load Llama-3-8B in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'

# You need HuggingFace access to Llama-3. If gated, set your token:
# from huggingface_hub import login; login('YOUR_HF_TOKEN_HERE')
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
tokenizer.pad_token = tokenizer.eos_token

print('Loading model in 4-bit (this takes ~3 min on T4)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    output_hidden_states=True,
    output_attentions=True,
    token=os.environ['HF_TOKEN'],
)
model.eval()
print('Model loaded. GPU memory:', torch.cuda.memory_allocated() / 1e9, 'GB')

In [ ]:
# Cell 5: Feature extraction
import numpy as np
from sklearn.decomposition import PCA

PROBE_LAYER = 16  # middle layer — best signal per WhiteBoxProbe design
MAX_TOKENS = 512

def extract_features(chain: dict) -> np.ndarray:
    """Extract [PCA(hidden_state_layer16), attention_entropy] from final token."""
    # Build chain text
    steps = chain.get('steps', [])
    parts = [f"Q: {chain.get('question', '')[:200]}"]
    for i, s in enumerate(steps[:6], 1):
        parts.append(f"Step {i}: {s.get('thought', '')[:100]}")
        if s.get('observation'):
            parts.append(f"Obs: {s.get('observation', '')[:80]}")
    parts.append(f"A: {chain.get('final_answer', '')}")
    text = ' '.join(parts)

    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=MAX_TOKENS, padding=False
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Hidden state at layer PROBE_LAYER, final token
    hs = outputs.hidden_states[PROBE_LAYER][0, -1, :].float().cpu().numpy()  # (4096,)

    # Attention entropy at layer PROBE_LAYER (mean over heads)
    attn = outputs.attentions[PROBE_LAYER][0]  # (n_heads, seq, seq)
    # Entropy of attention distribution at final position
    attn_last = attn[:, -1, :].float().cpu().numpy()  # (n_heads, seq)
    eps = 1e-10
    entropy_per_head = -np.sum(attn_last * np.log(attn_last + eps), axis=-1)  # (n_heads,)
    mean_entropy = float(np.mean(entropy_per_head))

    return hs, mean_entropy

print('Extracting features for all 500 chains...')
hidden_states = []
entropies = []
labels = []

for i, chain in enumerate(all_chains):
    try:
        hs, ent = extract_features(chain)
        hidden_states.append(hs)
        entropies.append(ent)
        labels.append(0 if chain.get('correct', True) else 1)
    except Exception as e:
        print(f'  Chain {i} failed: {e}')
        continue

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500]  GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB')

print(f'Extracted features for {len(labels)} chains')
print(f'Wrong: {sum(labels)}/{len(labels)}')

In [ ]:
# Cell 6: PCA + probe training
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings

X_hs  = np.array(hidden_states)   # (N, 4096)
X_ent = np.array(entropies).reshape(-1, 1)  # (N, 1)
y     = np.array(labels)

# Train/test 70/30
n_train = int(0.7 * len(y))
X_hs_tr,  X_hs_te  = X_hs[:n_train],  X_hs[n_train:]
X_ent_tr, X_ent_te = X_ent[:n_train], X_ent[n_train:]
y_tr, y_te          = y[:n_train],     y[n_train:]

# PCA on hidden states (fit on train only)
pca = PCA(n_components=64, random_state=42)
X_pca_tr = pca.fit_transform(X_hs_tr)
X_pca_te = pca.transform(X_hs_te)

# Two-track: PCA(hidden) + direct entropy — entropy must NOT go through PCA
X_tr = np.hstack([X_pca_tr, X_ent_tr])  # (n_train, 65)
X_te = np.hstack([X_pca_te, X_ent_te])  # (n_test, 65)

# LogReg probe
probe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(C=0.1, max_iter=1000, random_state=42)),
])
probe.fit(X_tr, y_tr)
probe_scores = probe.predict_proba(X_te)[:, 1]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    auroc = roc_auc_score(y_te, probe_scores)
auroc = max(auroc, 1.0 - auroc)

# Also entropy alone
auroc_ent = roc_auc_score(y_te, X_ent_te.ravel())
auroc_ent = max(auroc_ent, 1.0 - auroc_ent)

print(f'\nWhiteBoxProbe Results (MuSiQue n=500, 70/30 split)')
print(f'  Probe (PCA+entropy) AUROC: {auroc:.4f}')
print(f'  Entropy alone AUROC:       {auroc_ent:.4f}')
print(f'  n_train={n_train}  n_test={len(y_te)}  n_wrong_test={int(y_te.sum())}')

In [ ]:
# Cell 7: Bootstrap CI
rng = np.random.default_rng(42)
boot = []
pos_ix = np.where(y_te == 1)[0]
neg_ix = np.where(y_te == 0)[0]
for _ in range(2000):
    bi = np.concatenate([rng.choice(pos_ix, len(pos_ix), replace=True),
                         rng.choice(neg_ix, len(neg_ix), replace=True)])
    yb = y_te[bi]; sb = probe_scores[bi]
    if len(np.unique(yb)) < 2: continue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        auc = roc_auc_score(yb, sb)
    boot.append(max(auc, 1.0 - auc))
lo, hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
sig = lo > 0.5
print(f'  95% CI: [{lo:.3f}, {hi:.3f}]  {"✅ SIGNIFICANT" if sig else "⚠ not significant"}')

In [ ]:
# Cell 8: Save probe + results
import pickle

probe_bundle = {
    'pca': pca,
    'probe': probe,
    'layer': PROBE_LAYER,
    'auroc': float(auroc),
    'ci': [float(lo), float(hi)],
    'significant': bool(sig),
    'n_train': int(n_train),
    'n_test': int(len(y_te)),
    'domain': 'musique',
    'model': MODEL_NAME,
}

with open('white_box_probe_musique.pkl', 'wb') as f:
    pickle.dump(probe_bundle, f)

result = {
    'experiment': 'white_box_minijudge_musique',
    'model': MODEL_NAME,
    'domain': 'musique',
    'n_chains': len(labels),
    'n_wrong': int(sum(labels)),
    'probe_auroc': float(auroc),
    'entropy_auroc': float(auroc_ent),
    'ci_lo': float(lo),
    'ci_hi': float(hi),
    'significant': bool(sig),
}
with open('white_box_result.json', 'w') as f:
    json.dump(result, f, indent=2)

print('Saved: white_box_probe_musique.pkl')
print('Saved: white_box_result.json')
print(json.dumps(result, indent=2))

In [ ]:
# Cell 9: Download results
from google.colab import files
files.download('white_box_probe_musique.pkl')
files.download('white_box_result.json')
print('Downloaded. Place white_box_probe_musique.pkl in:')
print('  results/exp_white_box_musique/white_box_probe_musique.pkl')
print('Then run: python3 experiments/exp_white_box_musique_eval.py')